# TurAIsta Dataset Audit: MMS, FLORES+/NLLB-200, and FLEURS

This notebook verifies the three datasets selected for TurAIsta by loading them directly. It produces one file, `audit_out.zip`, which contains the computed coverage tables, five real samples per language, and basic quality statistics.

Before running, complete the setup in the next cell's instructions.

## Setup Checklist

1. In the right panel, open Session options and set Internet to On.
2. On huggingface.co, create a read token (Settings, then Access Tokens).
3. While logged in to Hugging Face, open https://huggingface.co/datasets/openlanguagedata/flores_plus and accept the terms.
4. In this notebook, open Add-ons, then Secrets. Add a secret named `HF_TOKEN` with your token, and enable it for this notebook.
5. Run all cells from top to bottom (Run All).

## 1. Install Dependencies

In [2]:
!pip install -q "datasets>=2.19,<4" "huggingface_hub>=0.23" soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 4.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2

## 2. Configuration and Login

The token is read from Kaggle Secrets. The cell stops early with a clear message if the token is missing.

In [3]:
import os, io, json, re, time, unicodedata, shutil
from collections import Counter
from pathlib import Path

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN secret is missing. See the setup checklist."
os.environ["HF_TOKEN"] = HF_TOKEN

OUT = Path("/kaggle/working/audit_out")
OUT.mkdir(parents=True, exist_ok=True)
N_SAMPLES = 5          # real rows shown per language
QUALITY_ROWS = 2000    # rows used for quality statistics

TARGET = ["tl", "ceb", "ilo", "en", "ko", "ja", "zh"]
HOST = {"tl", "ceb", "ilo"}

def save(name, obj):
    (OUT / f"{name}.json").write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str))
    print(f"saved {name}.json")

## 3. Quality Checks

These checks are applied to every text sample: duplicates, wrong script, encoding errors, personal data patterns, and, for host languages, a rough estimate of English code-switching (Taglish).

In [4]:
SCRIPTS = {"latin": [(0x41, 0x24F)], "hangul": [(0xAC00, 0xD7AF), (0x1100, 0x11FF)],
           "kana": [(0x3040, 0x30FF)], "han": [(0x4E00, 0x9FFF), (0x3400, 0x4DBF)]}
EXPECTED = {"tl": {"latin"}, "ceb": {"latin"}, "ilo": {"latin"}, "en": {"latin"},
            "ko": {"hangul"}, "ja": {"kana", "han"}, "zh": {"han"}}
PII = {"email": r"[\w.+-]+@[\w-]+\.[\w.-]+", "ph_mobile": r"(?:\+?63|0)9\d{2}[\s-]?\d{3}[\s-]?\d{4}",
       "url": r"https?://\S+|www\.\S+"}
EN_WORDS = set("the and is are you your to of for with this that it in on at can will what how where please thank".split())

def script_of(ch):
    cp = ord(ch)
    for name, ranges in SCRIPTS.items():
        if any(lo <= cp <= hi for lo, hi in ranges):
            return name

def norm(t):
    t = unicodedata.normalize("NFKC", t).lower()
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", t)).strip()

def quality(texts, lang):
    texts = [t for t in texts if isinstance(t, str) and t.strip()]
    n = len(texts)
    if not n:
        return {"n": 0}
    off = moj = 0; pii = Counter(); en = 0.0
    for t in texts:
        sc = Counter(s for s in map(script_of, t) if s)
        if sc and sum(v for k, v in sc.items() if k in EXPECTED[lang]) / sum(sc.values()) < 0.8:
            off += 1
        if re.search(r"Ã.|Â.|â€|\ufffd", t):
            moj += 1
        for k, rx in PII.items():
            if re.search(rx, t):
                pii[k] += 1
        if lang in HOST:
            w = norm(t).split()
            en += sum(x in EN_WORDS for x in w) / max(len(w), 1)
    lens = sorted(len(t.split()) for t in texts)
    out = {"n": n, "exact_dup_rate": round(1 - len(set(texts)) / n, 4),
           "normalized_dup_rate": round(1 - len({norm(t) for t in texts}) / n, 4),
           "off_script_rate": round(off / n, 4), "mojibake_rate": round(moj / n, 4),
           "pii_hits": dict(pii), "median_words": lens[n // 2]}
    if lang in HOST:
        out["english_word_ratio"] = round(en / n, 4)
    return out

## 4. MMS (Speech Recognition and Text-to-Speech)

MMS is released as models rather than a corpus, so this section checks which languages the released checkpoints support.

In [7]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
ISO3 = {"tl": "tgl", "ceb": "ceb", "ilo": "ilo", "en": "eng", "ko": "kor", "ja": "jpn", "zh": "cmn-script_simplified"}

asr_info = api.model_info("facebook/mms-1b-all")
asr_files = set(api.list_repo_files("facebook/mms-1b-all"))
mms = {"asr_repo": "facebook/mms-1b-all", "asr_sha": asr_info.sha, "coverage": {}}
for lang, iso in ISO3.items():
    tts_repo = f"facebook/mms-tts-{iso.split('-')[0]}"
    try:
        api.model_info(tts_repo); tts = tts_repo
    except Exception:
        tts = "not covered"
    mms["coverage"][lang] = {"asr_adapter": any(f.endswith(f".{iso}.safetensors") for f in asr_files),
                             "tts_model": tts}
save("mms", mms)
mms["coverage"]

saved mms.json


{'tl': {'asr_adapter': True, 'tts_model': 'facebook/mms-tts-tgl'},
 'ceb': {'asr_adapter': True, 'tts_model': 'facebook/mms-tts-ceb'},
 'ilo': {'asr_adapter': True, 'tts_model': 'facebook/mms-tts-ilo'},
 'en': {'asr_adapter': True, 'tts_model': 'facebook/mms-tts-eng'},
 'ko': {'asr_adapter': True, 'tts_model': 'facebook/mms-tts-kor'},
 'ja': {'asr_adapter': True, 'tts_model': 'not covered'},
 'zh': {'asr_adapter': True, 'tts_model': 'not covered'}}

## 5. FLORES+ and NLLB-200 (Translation)

FLORES+ is an evaluation set. Only `dev` rows are shown as samples, and `devtest` is counted but never displayed or used for anything else.

In [8]:
from datasets import load_dataset, Audio
from huggingface_hub import hf_hub_download

FLORES = "openlanguagedata/flores_plus"
FCODE = {"tl": "tgl_Latn", "ceb": "ceb_Latn", "ilo": "ilo_Latn", "en": "eng_Latn",
         "ko": "kor_Hang", "ja": "jpn_Jpan", "zh": "cmn_Hans"}
NLLB_CODE = {**FCODE, "zh": "zho_Hans"}

flores = {"repo": FLORES, "sha": api.dataset_info(FLORES).sha, "coverage": {}, "samples": {}, "quality": {}}
for lang, code in FCODE.items():
    try:
        dev = load_dataset(FLORES, code, split="dev", token=HF_TOKEN)
        devtest = load_dataset(FLORES, code, split="devtest", token=HF_TOKEN)
    except Exception as e:
        flores["coverage"][lang] = f"not covered or error: {e!r}"[:300]
        continue
    flores["coverage"][lang] = {"config": code, "dev": len(dev), "devtest": len(devtest)}
    flores["samples"][lang] = [{"id": r["id"], "text": r["text"], "topic": r.get("topic")}
                               for r in dev.select(range(N_SAMPLES))]
    flores["quality"][lang] = quality(dev["text"], lang)

tok = json.load(open(hf_hub_download("facebook/nllb-200-distilled-600M", "special_tokens_map.json")))
nllb_langs = set(tok.get("additional_special_tokens", []))
flores["nllb200_supports"] = {lang: c in nllb_langs for lang, c in NLLB_CODE.items()}
save("flores_nllb", flores)
flores["coverage"], flores["nllb200_supports"]

Resolving data files:   0%|          | 0/228 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/222 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/228 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/222 [00:00<?, ?it/s]

ceb_Latn.jsonl:   0%|          | 0.00/448k [00:00<?, ?B/s]

...

Generating devtest split:   0%|          | 0/1012 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/228 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/222 [00:00<?, ?it/s]

saved flores_nllb.json


({'tl': 'not covered or error: ValueError("BuilderConfig \'tgl_Latn\' not found. Available: [\'default\', \'ace_Arab\', \'ace_Latn\', \'acm_Arab\', \'acq_Arab\', \'aeb_Arab\', \'afr_Latn\', \'als_Latn\', \'amh_Ethi\', \'apc_Arab_nort3139\', \'apc_Arab_sout3123\', \'apd_Arab\', \'arb_Arab\', \'arb_Latn\', \'arg_Latn\', \'ars_Arab\', \'ary_Arab',
  'ceb': {'config': 'ceb_Latn', 'dev': 997, 'devtest': 1012},
  'ilo': {'config': 'ilo_Latn', 'dev': 997, 'devtest': 1012},
  'en': {'config': 'eng_Latn', 'dev': 997, 'devtest': 1012},
  'ko': {'config': 'kor_Hang', 'dev': 997, 'devtest': 1012},
  'ja': {'config': 'jpn_Jpan', 'dev': 997, 'devtest': 1012},
  'zh': {'config': 'cmn_Hans', 'dev': 997, 'devtest': 1012}},
 {'tl': True,
  'ceb': True,
  'ilo': True,
  'en': True,
  'ko': True,
  'ja': True,
  'zh': True})

## 6. FLEURS (Speech Benchmark)

Audio is streamed and never fully decoded. Hours are computed from each file's header. This is the slowest section; set `FLEURS_LIMIT` to a small number (for example 50) for a quick test, or `None` for exact counts.

In [10]:
import csv
from huggingface_hub import hf_hub_download

FL = "google/fleurs"
FLCODE = {"tl": "fil_ph", "ceb": "ceb_ph", "ilo": None, "en": "en_us",
          "ko": "ko_kr", "ja": "ja_jp", "zh": "cmn_hans_cn"}
SR = 16000  # FLEURS audio sample rate

def read_tsv(code, split):
    for name in ([split] if split != "validation" else ["dev", "validation"]):
        try:
            p = hf_hub_download(FL, f"data/{code}/{name}.tsv", repo_type="dataset")
            with open(p, encoding="utf-8") as f:
                return list(csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE))
        except Exception as e:
            err = e
    raise err

fleurs = {"repo": FL, "sha": api.dataset_info(FL).sha, "coverage": {}, "samples": {}, "quality": {}}
for lang, code in FLCODE.items():
    if code is None:
        fleurs["coverage"][lang] = "not covered"; continue
    fleurs["coverage"][lang] = {"config": code}
    for split in ["train", "validation", "test"]:
        rows = read_tsv(code, split)
        # columns: id, file_name, raw_transcription, transcription, characters, num_samples, gender
        secs = sum(int(r[5]) for r in rows if len(r) > 5 and r[5].isdigit()) / SR
        fleurs["coverage"][lang][split] = {"utterances": len(rows),
                                           "unique_sentences": len({r[0] for r in rows}),
                                           "hours": round(secs / 3600, 2)}
        if split == "train":
            fleurs["samples"][lang] = [{"id": r[0], "text": r[3], "raw": r[2],
                                        "seconds": round(int(r[5]) / SR, 2), "gender": r[6]}
                                       for r in rows[:N_SAMPLES]]
            fleurs["quality"][lang] = quality([r[3] for r in rows][:QUALITY_ROWS], lang)
    print(lang, fleurs["coverage"][lang])
save("fleurs", fleurs)

train.tsv:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/278k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/676k [00:00<?, ?B/s]

tl {'config': 'fil_ph', 'train': {'utterances': 1884, 'unique_sentences': 1219, 'hours': 7.71}, 'validation': {'utterances': 418, 'unique_sentences': 150, 'hours': 1.98}, 'test': {'utterances': 964, 'unique_sentences': 349, 'hours': 4.79}}


train.tsv:   0%|          | 0.00/2.09M [00:00<?, ?B/s]

...

ja {'config': 'ja_jp', 'train': {'utterances': 2292, 'unique_sentences': 1332, 'hours': 7.42}, 'validation': {'utterances': 266, 'unique_sentences': 135, 'hours': 0.94}, 'test': {'utterances': 650, 'unique_sentences': 321, 'hours': 2.36}}


train.tsv:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/205k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/491k [00:00<?, ?B/s]

zh {'config': 'cmn_hans_cn', 'train': {'utterances': 3246, 'unique_sentences': 1500, 'hours': 9.73}, 'validation': {'utterances': 409, 'unique_sentences': 150, 'hours': 1.27}, 'test': {'utterances': 945, 'unique_sentences': 349, 'hours': 3.07}}
saved fleurs.json


## 7. Package the Results

Download `audit_out.zip` from the Output panel on the right and share it.

In [13]:
import os
from IPython.display import FileLink
os.chdir("/kaggle/working")
print(os.listdir("/kaggle/working"))
print(os.listdir("/kaggle/working/audit_out"))
FileLink("audit_out.zip")

['audit_out', '.virtual_documents', 'audit_out.zip']
['flores_nllb.json', 'fleurs.json', 'mms.json']


/kaggle/working/audit_out.zip